In [1]:
import serial

In [2]:
ser = serial.Serial()
ser.baudrate = 115200
ser.port = "COM4"
ser.timeout = 3

In [3]:
ser.open()
# ser.isOpen()
# ser.close()

In [34]:
reqAddrAndData = [
    # ARP Probe for 169.254.10.10 (one of Link Local Addresses)
    # with AXI Ethernet Lite MAC IP Core.
    # 
    # dest. addr. = FF:FF:FF:FF:FF:FF (broadcast)
    # src. addr. = 00:00:5E:00:FA:CE (default in userguide pg135-axi-ethernetlite)
    (0x0000, 0xFFFF_FFFF),
    (0x0004, 0x0000_FFFF),
    (0x0008, 0xCEFA_005E),
    # EtherType = 0x0806 (ARP)
    # ethernet frame (ARP packet) payload starts here
    # HTYPE = 1
    (0x000C, 0x0100_0608),
    # PTYPE = 0x0800,
    # HLEN (MAC addr. length) = 6,
    # PLEN (IP addr. length) = 4
    (0x0010, 0x0406_0008),
    # OPERATION = 1 (Request)
    # Sender MAC = 00:00:5E:00:FA:CE
    (0x0014, 0x0000_0100),
    (0x0018, 0xCEFA_005E),
    # Sender IP = 0.0.0.0 (RFC 3927)
    (0x001C, 0x0000_0000),
    # Target MAC = 00:00:00:00:00:00 (RFC 3927)
    (0x0020, 0x0000_0000),
    # Target IP = 169.254.114.185
    (0x0024, 0xFEA9_0000),
    (0x0028, (185 << 8) | 114),
    
    # Length of packet at 0x07F4 (AXI Ethernet Lite MAC)
    # sender/target MAC addr. (6 + 6), ethertype (2), and payload (28)
    (0x07F4, 14 + 28),

    # Set Transmit Status at 0x07FC
    (0x07FC, 0b1),
]

# addr = 0x9988
# data = 0x00112233
encBuffer = []
for addr, data in reqAddrAndData:
    addrBytes = addr.to_bytes(2, "little")
    dataBytes = data.to_bytes(4, "little")
# addrBytes, dataBytes

# data = [addrBytes, dataBytes]
    enc = addrBytes + dataBytes
    encBuffer.append(enc)

for d in encBuffer:
    # print(d)
    ser.write(d)
# for d in data:
#     ser.write(d)
#     # print(d)
# print(enc)

In [33]:
for _ in reqAddrAndData:
    s = ser.readline()
    print(s)

b'0000000000000027 0000 FFFFFFFF\r\n'
b'0000000000000028 0004 0000FFFF\r\n'
b'0000000000000029 0008 CEFA005E\r\n'
b'000000000000002A 000C 01000608\r\n'
b'000000000000002B 0010 04060008\r\n'
b'000000000000002C 0014 00000100\r\n'
b'000000000000002D 0018 CEFA005E\r\n'
b'000000000000002E 001C 00000000\r\n'
b'000000000000002F 0020 00000000\r\n'
b'0000000000000030 0024 FEA90000\r\n'
b'0000000000000031 0028 0000B872\r\n'
b'0000000000000032 07F4 0000002A\r\n'
b'0000000000000033 07FC 00000001\r\n'


In [22]:
ser.readline()

b''